# Classification Rule Learning using CN2 and FOIL

**Dataset:** Kaggle `PlayTennis.csv` dataset  
**Kaggle source:** https://www.kaggle.com/datasets/sdk1810/playtennis

## Aim
1. Read a Kaggle classification dataset.
2. Use an example-based rule learner (**CN2-style sequential covering**) to generate classification rules.
3. Apply the **FOIL (First-Order Inductive Learner)** information-gain principle to learn first-order rules for prediction.

> The Kaggle dataset is the classic 14-example *Play Tennis* dataset. Because it is intentionally tiny and used for rule-learning demonstrations, this notebook learns rules from all 14 examples and reports **training/resubstitution accuracy**. This is for demonstrating the algorithms, not estimating real-world generalization performance.


In [1]:
import math
import pandas as pd
from io import StringIO
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

pd.set_option("display.max_columns", None)

## 1. Read the Kaggle dataset

The CSV below reproduces the classic `PlayTennis.csv` data hosted on Kaggle. Keeping it inside the notebook makes the notebook completely self-contained and executable without Kaggle credentials.

In [2]:
PLAY_TENNIS_CSV = r"""Outlook,Temperature,Humidity,Wind,PlayTennis
Sunny,Hot,High,Weak,No
Sunny,Hot,High,Strong,No
Overcast,Hot,High,Weak,Yes
Rain,Mild,High,Weak,Yes
Rain,Cool,Normal,Weak,Yes
Rain,Cool,Normal,Strong,No
Overcast,Cool,Normal,Strong,Yes
Sunny,Mild,High,Weak,No
Sunny,Cool,Normal,Weak,Yes
Rain,Mild,Normal,Weak,Yes
Sunny,Mild,Normal,Strong,Yes
Overcast,Mild,High,Strong,Yes
Overcast,Hot,Normal,Weak,Yes
Rain,Mild,High,Strong,No
"""

# Read the data exactly as a CSV dataset.
df = pd.read_csv(StringIO(PLAY_TENNIS_CSV))
print("Dataset shape:", df.shape)
df

Dataset shape: (14, 5)


,Outlook,Temperature,Humidity,Wind,PlayTennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes
5,Rain,Cool,Normal,Strong,No
6,Overcast,Cool,Normal,Strong,Yes
7,Sunny,Mild,High,Weak,No
8,Sunny,Cool,Normal,Weak,Yes
9,Rain,Mild,Normal,Weak,Yes


In [3]:
print("Class distribution:")
print(df["PlayTennis"].value_counts())

print("\nUnique values per feature:")
for c in df.columns[:-1]:
    print(f"{c:12s}: {sorted(df[c].unique())}")

Class distribution:
PlayTennis
Yes    9
No     5
Name: count, dtype: int64

Unique values per feature:
Outlook     : ['Overcast', 'Rain', 'Sunny']
Temperature : ['Cool', 'Hot', 'Mild']
Humidity    : ['High', 'Normal']
Wind        : ['Strong', 'Weak']


## 2. CN2-style example-based rule learning

CN2 is a **separate-and-conquer / sequential-covering** rule learner. It repeatedly searches for a conjunction of feature-value tests that covers many positive examples and few negative examples, stores that rule, removes the positive examples it explains, and repeats.

This implementation uses a small **beam search** and scores candidate complexes using **Laplace accuracy**:

\[
\text{Laplace}(R)=\frac{p+1}{p+n+2}
\]

where \(p\) and \(n\) are the numbers of positive and negative examples covered by rule \(R\).


In [4]:
TARGET = "PlayTennis"
POSITIVE = "Yes"
NEGATIVE = "No"
FEATURES = [c for c in df.columns if c != TARGET]

def rule_mask(data, rule):
    """Return the rows covered by a conjunction of (feature, value) literals."""
    mask = pd.Series(True, index=data.index)
    for feature, value in rule:
        mask &= data[feature].eq(value)
    return mask

def rule_to_text(rule, head="PlayTennis=Yes"):
    body = " AND ".join(f"{f}={v}" for f, v in rule) if rule else "TRUE"
    return f"IF {body} THEN {head}"

def cn2_learn(data, target=TARGET, positive=POSITIVE, beam_width=10, max_rule_len=3):
    features = [c for c in data.columns if c != target]
    literals = [(f, v) for f in features for v in sorted(data[f].unique())]
    remaining_positive = set(data.index[data[target].eq(positive)])
    learned_rules = []

    while remaining_positive:
        beam = [tuple()]
        seen = set()
        best_rule = None
        best_key = None

        for _depth in range(1, max_rule_len + 1):
            candidates = []
            for rule in beam:
                used_features = {f for f, _ in rule}
                for literal in literals:
                    if literal[0] in used_features:
                        continue
                    new_rule = tuple(sorted(rule + (literal,)))
                    if new_rule in seen:
                        continue
                    seen.add(new_rule)

                    mask = rule_mask(data, new_rule)
                    pos_mask = data.index.to_series().isin(remaining_positive)
                    p = int((mask & pos_mask).sum())
                    n = int((mask & ~data[target].eq(positive)).sum())
                    if p == 0:
                        continue

                    laplace = (p + 1) / (p + n + 2)
                    # Prefer high Laplace score, then more positives, then fewer negatives.
                    key = (laplace, p, -n, -len(new_rule))
                    candidates.append((key, new_rule))
                    if best_key is None or key > best_key:
                        best_key, best_rule = key, new_rule

            candidates.sort(key=lambda x: x[0], reverse=True)
            beam = [r for _, r in candidates[:beam_width]]
            if not beam:
                break

        if best_rule is None:
            break

        covered = rule_mask(data, best_rule)
        covered_positive = {i for i in remaining_positive if bool(covered.loc[i])}
        if not covered_positive:
            break

        learned_rules.append(best_rule)
        remaining_positive -= covered_positive

    return learned_rules

def predict_with_rules(data, rules, positive=POSITIVE, negative=NEGATIVE):
    predictions = []
    for _, row in data.iterrows():
        matched = any(all(row[f] == v for f, v in rule) for rule in rules)
        predictions.append(positive if matched else negative)
    return predictions

In [5]:
cn2_rules = cn2_learn(df)

print("CN2-style learned rules:\n")
for i, rule in enumerate(cn2_rules, 1):
    print(f"R{i}: {rule_to_text(rule)}")
print("DEFAULT: PlayTennis=No")

CN2-style learned rules:

R1: IF Outlook=Overcast THEN PlayTennis=Yes
R2: IF Humidity=Normal AND Wind=Weak THEN PlayTennis=Yes
R3: IF Humidity=Normal AND Temperature=Mild THEN PlayTennis=Yes
R4: IF Outlook=Rain AND Wind=Weak THEN PlayTennis=Yes
DEFAULT: PlayTennis=No


In [6]:
cn2_pred = predict_with_rules(df, cn2_rules)
print("CN2 training accuracy:", accuracy_score(df[TARGET], cn2_pred))
print("\nConfusion matrix [No, Yes]:")
print(confusion_matrix(df[TARGET], cn2_pred, labels=["No", "Yes"]))
print("\nClassification report:")
print(classification_report(df[TARGET], cn2_pred, zero_division=0))

CN2 training accuracy: 1.0

Confusion matrix [No, Yes]:
[[5 0]
 [0 9]]

Classification report:
              precision    recall  f1-score   support

          No       1.00      1.00      1.00         5
         Yes       1.00      1.00      1.00         9

    accuracy                           1.00        14
   macro avg       1.00      1.00      1.00        14
weighted avg       1.00      1.00      1.00        14



## 3. FOIL: First-Order Inductive Learner

FOIL learns clauses by specializing a general rule with literals that improve its ability to separate positive from negative examples.

Its classic information-gain measure is:

\[
\text{FOILGain}(L)=t\left[\log_2\left(\frac{p_1}{p_1+n_1}\right)-\log_2\left(\frac{p_0}{p_0+n_0}\right)\right]
\]

For this tabular dataset, each row is treated as an entity \(X\), and each categorical feature becomes a first-order predicate. For example:

```prolog
outlook(x, overcast).
humidity(x, normal).
wind(x, weak).
play_tennis(x).
```

A learned clause can therefore be written as:

```prolog
play_tennis(X) :- outlook(X, overcast).
```

This is a unary-relational specialization of FOIL: the learner still adds first-order literals using FOIL gain, while the relational database is constructed from the table's categorical predicates.


In [7]:
def foil_gain(p0, n0, p1, n1, t):
    if p0 <= 0 or p1 <= 0:
        return float("-inf")
    old_precision = p0 / (p0 + n0)
    new_precision = p1 / (p1 + n1)
    if old_precision <= 0 or new_precision <= 0:
        return float("-inf")
    return t * (math.log2(new_precision) - math.log2(old_precision))

def foil_learn(data, target=TARGET, positive=POSITIVE, max_rules=10, max_rule_len=4):
    features = [c for c in data.columns if c != target]
    literals = [(f, v) for f in features for v in sorted(data[f].unique())]
    uncovered_positive = set(data.index[data[target].eq(positive)])
    rules = []

    for _ in range(max_rules):
        if not uncovered_positive:
            break

        rule = []
        current_mask = pd.Series(True, index=data.index)

        while len(rule) < max_rule_len:
            positive_pool = data.index.to_series().isin(uncovered_positive)
            p0 = int((current_mask & positive_pool).sum())
            n0 = int((current_mask & ~data[target].eq(positive)).sum())

            if p0 == 0 or n0 == 0:
                break

            best_literal = None
            best_mask = None
            best_key = None

            used_features = {f for f, _ in rule}
            for literal in literals:
                feature, value = literal
                if feature in used_features:
                    continue

                new_mask = current_mask & data[feature].eq(value)
                p1 = int((new_mask & positive_pool).sum())
                n1 = int((new_mask & ~data[target].eq(positive)).sum())
                if p1 == 0:
                    continue

                gain = foil_gain(p0, n0, p1, n1, t=p1)
                key = (gain, -n1, p1)
                if best_key is None or key > best_key:
                    best_key = key
                    best_literal = literal
                    best_mask = new_mask

            if best_literal is None or best_key[0] <= 0:
                break

            rule.append(best_literal)
            current_mask = best_mask

        if not rule:
            break

        covered = rule_mask(data, rule)
        covered_positive = {i for i in uncovered_positive if bool(covered.loc[i])}
        if not covered_positive:
            break

        rules.append(tuple(rule))
        uncovered_positive -= covered_positive

    return rules

def foil_rule_to_prolog(rule):
    def pred_name(feature):
        return feature.lower().replace("-", "_").replace(" ", "_")
    body = ", ".join(f"{pred_name(f)}(X, {str(v).lower()})" for f, v in rule)
    return f"play_tennis(X) :- {body}."

In [8]:
foil_rules = foil_learn(df)

print("FOIL learned first-order rules:\n")
for i, rule in enumerate(foil_rules, 1):
    print(f"F{i}: {foil_rule_to_prolog(rule)}")
print("DEFAULT: not play_tennis(X).")

FOIL learned first-order rules:

F1: play_tennis(X) :- outlook(X, overcast).
F2: play_tennis(X) :- humidity(X, normal), wind(X, weak).
F3: play_tennis(X) :- temperature(X, mild), humidity(X, normal).
F4: play_tennis(X) :- outlook(X, rain), wind(X, weak).
DEFAULT: not play_tennis(X).


In [9]:
foil_pred = predict_with_rules(df, foil_rules)
print("FOIL training accuracy:", accuracy_score(df[TARGET], foil_pred))
print("\nConfusion matrix [No, Yes]:")
print(confusion_matrix(df[TARGET], foil_pred, labels=["No", "Yes"]))
print("\nClassification report:")
print(classification_report(df[TARGET], foil_pred, zero_division=0))

FOIL training accuracy: 1.0

Confusion matrix [No, Yes]:
[[5 0]
 [0 9]]

Classification report:
              precision    recall  f1-score   support

          No       1.00      1.00      1.00         5
         Yes       1.00      1.00      1.00         9

    accuracy                           1.00        14
   macro avg       1.00      1.00      1.00        14
weighted avg       1.00      1.00      1.00        14



## 4. Final comparison

Both algorithms produce interpretable rules. CN2 expresses the result as ordinary IF-THEN classification rules, while FOIL expresses essentially the same learned structure as first-order predicates over an entity `X`.

In [10]:
summary = pd.DataFrame({
    "Method": ["CN2-style", "FOIL"],
    "Number of rules": [len(cn2_rules), len(foil_rules)],
    "Training accuracy": [
        accuracy_score(df[TARGET], cn2_pred),
        accuracy_score(df[TARGET], foil_pred),
    ],
})
summary

,Method,Number of rules,Training accuracy
0,CN2-style,4,1.0
1,FOIL,4,1.0


## Conclusion

- **CN2-style learning** generated compact human-readable classification rules from examples using sequential covering and beam search.
- **FOIL** specialized rules using FOIL information gain and expressed the result as first-order predicates such as `outlook(X, overcast)`.
- On this classic 14-row demonstration dataset, both learned rule sets classify all training examples correctly.
- Since the dataset is tiny, the reported 100% value is **training accuracy only** and should not be interpreted as a real-world performance estimate.